In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%whos

Variable   Type      Data/Info
------------------------------
np         module    <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
pd         module    <module 'pandas' from '/h<...>ages/pandas/__init__.py'>
plt        module    <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
sns        module    <module 'seaborn' from '/<...>ges/seaborn/__init__.py'>


### Load data

In [13]:
URL = r"/mnt/c/users/leduc/Downloads/archive/"
pos_cash_balance = pd.read_csv(URL+"POS_CASH_balance.csv")
application_train = pd.read_csv(URL+"application_train.csv")
bureau = pd.read_csv(URL+"bureau.csv")
bureau_balance= pd.read_csv(URL+"bureau_balance.csv")
credit_card_balance= pd.read_csv(URL+"credit_card_balance.csv")
installments_payments = pd.read_csv(URL+"installments_payments.csv")
previous_application = pd.read_csv(URL+"previous_application.csv")

In [21]:
print("pos_cash_balance", pos_cash_balance.shape)
print("application_train", application_train.shape)
print("bureau", bureau.shape)
print("bureau_balance", bureau_balance.shape)
print("credit_card_balance",credit_card_balance.shape)
print("installments_payments",installments_payments.shape)
print("previous_application",previous_application.shape)

pos_cash_balance (10001358, 8)
application_train (307511, 122)
bureau (1716428, 17)
bureau_balance (27299925, 3)
credit_card_balance (3840312, 23)
installments_payments (13605401, 8)
previous_application (1670214, 37)


In [9]:
print("pos_cash_balance", pos_cash_balance['SK_ID_CURR'].nunique())
print("application_train", application_train['SK_ID_CURR'].nunique())
print("bureau", bureau['SK_ID_CURR'].nunique())
print("credit_card_balance",credit_card_balance['SK_ID_CURR'].nunique())
print("installments_payments",installments_payments['SK_ID_CURR'].nunique())
print("previous_application",previous_application['SK_ID_CURR'].nunique())

pos_cash_balance 337252
application_train 307511
bureau 305811
credit_card_balance 103558
installments_payments 339587
previous_application 338857


In [13]:
pos_cash_balance.groupby('SK_ID_CURR',as_index=False)['SK_ID_PREV'].count()

,SK_ID_CURR,SK_ID_PREV
0,100001,9
1,100002,19
2,100003,28
3,100004,4
4,100005,11
...,...,...
337247,456251,9
337248,456252,7
337249,456253,17
337250,456254,20


In [15]:
pos_cash_balance.loc[pos_cash_balance['SK_ID_CURR'] == 100001,:]

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
1261679,1851984,100001,-96,4.0,2.0,Active,0,0
1891462,1851984,100001,-95,4.0,1.0,Active,7,7
2197888,1369693,100001,-53,4.0,0.0,Completed,0,0
4704415,1369693,100001,-54,4.0,1.0,Active,0,0
4928574,1851984,100001,-93,4.0,0.0,Completed,0,0
7167007,1369693,100001,-57,4.0,4.0,Active,0,0
7823681,1369693,100001,-55,4.0,2.0,Active,0,0
8531326,1851984,100001,-94,4.0,0.0,Active,0,0
8789081,1369693,100001,-56,4.0,3.0,Active,0,0


In [17]:
bureau.groupby('SK_ID_CURR',as_index=False)['SK_ID_BUREAU'].count()

,SK_ID_CURR,SK_ID_BUREAU
0,100001,7
1,100002,8
2,100003,4
3,100004,2
4,100005,3
...,...,...
305806,456249,13
305807,456250,3
305808,456253,4
305809,456254,1


In [18]:
bureau.loc[bureau['SK_ID_CURR'] == 100001,:]

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
248484,100001,5896630,Closed,currency 1,-857,0,-492.0,-553.0,NaN,0,112500.0,0.0,0.0,0.0,Consumer credit,-155,0.0
248485,100001,5896631,Closed,currency 1,-909,0,-179.0,-877.0,NaN,0,279720.0,0.0,0.0,0.0,Consumer credit,-155,0.0
248486,100001,5896632,Closed,currency 1,-879,0,-514.0,-544.0,NaN,0,91620.0,0.0,0.0,0.0,Consumer credit,-155,0.0
248487,100001,5896633,Closed,currency 1,-1572,0,-1329.0,-1328.0,NaN,0,85500.0,0.0,0.0,0.0,Consumer credit,-155,0.0
248488,100001,5896634,Active,currency 1,-559,0,902.0,NaN,NaN,0,337680.0,113166.0,0.0,0.0,Consumer credit,-6,4630.5
248489,100001,5896635,Active,currency 1,-49,0,1778.0,NaN,NaN,0,378000.0,373239.0,0.0,0.0,Consumer credit,-16,10822.5
248490,100001,5896636,Active,currency 1,-320,0,411.0,NaN,NaN,0,168345.0,110281.5,NaN,0.0,Consumer credit,-10,9364.5


In [23]:
bureau_balance.groupby('SK_ID_BUREAU')['SK_ID_BUREAU'].count()

SK_ID_BUREAU
5001709    97
5001710    83
5001711     4
5001712    19
5001713    22
           ..
6842884    48
6842885    24
6842886    33
6842887    37
6842888    62
Name: SK_ID_BUREAU, Length: 817395, dtype: int64

In [14]:
acc_60dpd_l12m = bureau_balance\
                .loc[(
                (bureau_balance['MONTHS_BALANCE'] >= -12 )
                &(~bureau_balance['STATUS'].isin(['X','C','0','1','2'])
                )
        ),['SK_ID_BUREAU']]
acc_60dpd_l12m['acc_60dpd_l12m'] = 1

In [15]:
bureau = pd.merge(bureau,acc_60dpd_l12m,on='SK_ID_BUREAU',how='left')

In [16]:
bureau

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,acc_60dpd_l12m
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.00,0.0,NaN,0.0,Consumer credit,-131,NaN,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.00,171342.0,NaN,0.0,Credit card,-20,NaN,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.50,NaN,NaN,0.0,Consumer credit,-16,NaN,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.00,NaN,NaN,0.0,Credit card,-16,NaN,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.00,NaN,NaN,0.0,Consumer credit,-21,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1727428,259355,5057750,Active,currency 1,-44,0,-30.0,NaN,0.0,0,11250.00,11250.0,0.0,0.0,Microloan,-19,NaN,NaN
1727429,100044,5057754,Closed,currency 1,-2648,0,-2433.0,-2493.0,5476.5,0,38130.84,0.0,0.0,0.0,Consumer credit,-2493,NaN,NaN
1727430,100044,5057762,Closed,currency 1,-1809,0,-1628.0,-970.0,NaN,0,15570.00,NaN,NaN,0.0,Consumer credit,-967,NaN,NaN
1727431,246829,5057770,Closed,currency 1,-1878,0,-1513.0,-1513.0,NaN,0,36000.00,0.0,0.0,0.0,Consumer credit,-1508,NaN,NaN


In [61]:
bureau.columns = bureau.columns.str.lower()
bureau_features = bureau\
                .groupby('sk_id_curr',as_index=False)\
                .agg(
                    avg_days_credit = ('days_credit','mean'),
                    avg_credit_day_ovd = ('credit_day_overdue','mean'),
                    avg_credit_sum = ('amt_credit_sum','mean'),
                    max_credit_ovd = ('amt_credit_max_overdue','max'),
                    total_credit_sum = ('amt_credit_sum','sum'),
                    avg_credit_debt = ('amt_credit_sum_debt','mean'),
                    total_credit_debt = ('amt_credit_sum_debt','sum'),
                    avg_credit_ovd = ('amt_credit_sum_overdue','mean'),
                    toal_credit_ovd = ('amt_credit_sum_overdue','sum'),
                    total_bureau_acc =  ('sk_id_bureau','count'),
                    total_60dpd_l12m = ('acc_60dpd_l12m','count'),
                    
                )

In [62]:
credit_types = pd.get_dummies(bureau[['sk_id_curr','credit_type']],columns=['credit_type']).astype(int)

In [63]:
credit_types.groupby('sk_id_curr',as_index=False).sum()

,sk_id_curr,credit_type_Another type of loan,credit_type_Car loan,credit_type_Cash loan (non-earmarked),credit_type_Consumer credit,credit_type_Credit card,credit_type_Interbank credit,credit_type_Loan for business development,credit_type_Loan for purchase of shares (margin lending),credit_type_Loan for the purchase of equipment,credit_type_Loan for working capital replenishment,credit_type_Microloan,credit_type_Mobile operator loan,credit_type_Mortgage,credit_type_Real estate loan,credit_type_Unknown type of loan
0,100001,0,0,0,7,0,0,0,0,0,0,0,0,0,0,0
1,100002,0,0,0,4,4,0,0,0,0,0,0,0,0,0,0
2,100003,0,0,0,2,2,0,0,0,0,0,0,0,0,0,0
3,100004,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0
4,100005,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305806,456249,1,0,0,9,3,0,0,0,0,0,0,0,0,0,0
305807,456250,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0
305808,456253,0,0,0,3,1,0,0,0,0,0,0,0,0,0,0
305809,456254,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [65]:
bureau_features

,sk_id_curr,avg_days_credit,avg_credit_day_ovd,avg_credit_sum,max_credit_ovd,total_credit_sum,avg_credit_debt,total_credit_debt,avg_credit_ovd,toal_credit_ovd,total_bureau_acc,total_60dpd_l12m
0,100001,-735.000000,0.0,2.076236e+05,NaN,1453365.000,85240.928571,596686.500,0.0,0.0,7,0
1,100002,-874.000000,0.0,1.081319e+05,5043.645,865055.565,49156.200000,245781.000,0.0,0.0,8,0
2,100003,-1400.750000,0.0,2.543501e+05,0.000,1017400.500,0.000000,0.000,0.0,0.0,4,0
3,100004,-867.000000,0.0,9.451890e+04,0.000,189037.800,0.000000,0.000,0.0,0.0,2,0
4,100005,-190.666667,0.0,2.190420e+05,0.000,657126.000,189469.500000,568408.500,0.0,0.0,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...
305806,456249,-1667.076923,0.0,2.841430e+05,18945.000,3693858.660,16307.100000,163071.000,0.0,0.0,13,0
305807,456250,-862.000000,0.0,1.028820e+06,0.000,3086459.550,744013.365000,2232040.095,0.0,0.0,3,0
305808,456253,-867.500000,0.0,9.900000e+05,NaN,3960000.000,448958.250000,1795833.000,0.0,0.0,4,0
305809,456254,-1104.000000,0.0,4.500000e+04,NaN,45000.000,0.000000,0.000,0.0,0.0,1,0


In [66]:
bureau_features = bureau_features.merge(
    credit_types,
    on='sk_id_curr',
    how='left'
)

In [67]:
bureau_features

,sk_id_curr,avg_days_credit,avg_credit_day_ovd,avg_credit_sum,max_credit_ovd,total_credit_sum,avg_credit_debt,total_credit_debt,avg_credit_ovd,toal_credit_ovd,...,credit_type_Interbank credit,credit_type_Loan for business development,credit_type_Loan for purchase of shares (margin lending),credit_type_Loan for the purchase of equipment,credit_type_Loan for working capital replenishment,credit_type_Microloan,credit_type_Mobile operator loan,credit_type_Mortgage,credit_type_Real estate loan,credit_type_Unknown type of loan
0,100001,-735.000000,0.0,207623.571429,NaN,1453365.0,85240.928571,596686.50,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,100001,-735.000000,0.0,207623.571429,NaN,1453365.0,85240.928571,596686.50,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,100001,-735.000000,0.0,207623.571429,NaN,1453365.0,85240.928571,596686.50,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,100001,-735.000000,0.0,207623.571429,NaN,1453365.0,85240.928571,596686.50,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,100001,-735.000000,0.0,207623.571429,NaN,1453365.0,85240.928571,596686.50,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1727428,456255,-1089.454545,0.0,345629.045455,25578.0,3801919.5,191864.126250,1534913.01,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1727429,456255,-1089.454545,0.0,345629.045455,25578.0,3801919.5,191864.126250,1534913.01,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1727430,456255,-1089.454545,0.0,345629.045455,25578.0,3801919.5,191864.126250,1534913.01,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1727431,456255,-1089.454545,0.0,345629.045455,25578.0,3801919.5,191864.126250,1534913.01,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [38]:
credit_types['sk_id_curr'].duplicated().sum()

np.int64(1421622)

In [54]:
credit_types.index.is_unique

True